# 04. Statistical Analysis
**Project:** Logistics & Delivery Delays Analysis (Olist)
**Focus:** Hypothesis testing specifically targeted at the D1–D4 dashboard requirements: geographic influence, seller accountability, ETA accuracy, and customer satisfaction damage.


In [25]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import warnings
warnings.filterwarnings('ignore')

PATH_CLEANED = "../data/processed/olist_cleaned_data.csv"
df = pd.read_csv(PATH_CLEANED)
df['is_bad_review'] = df['review_score'] == 1
print(f"Dataset loaded. Shape: {df.shape}")


Dataset loaded. Shape: (109657, 44)


## 1. D4 Alignment: T-Test for Customer Impact
**Question:** Do late orders have statistically worse review scores than on-time orders?
**Dashboard:** D4 (Customer Experience Impact)


In [26]:
late_reviews   = df[df['is_late'] == True]['review_score'].dropna()
ontime_reviews = df[df['is_late'] == False]['review_score'].dropna()

t_stat, t_pval = stats.ttest_ind(ontime_reviews, late_reviews, equal_var=False)

print(f"T-Test (Review Scores: On-Time vs Late)")
print(f"  On-Time avg  : {ontime_reviews.mean():.2f}")
print(f"  Late avg     : {late_reviews.mean():.2f}")
print(f"  T-Statistic  : {t_stat:.4f}")
print(f"  P-Value      : {t_pval:.4e}")
if t_pval < 0.05:
    print("  Conclusion   : REJECT H0. Late deliveries significantly degrade customer satisfaction.")
else:
    print("  Conclusion   : Fail to reject H0.")


T-Test (Review Scores: On-Time vs Late)
  On-Time avg  : 4.21
  Late avg     : 2.48
  T-Statistic  : 89.1575
  P-Value      : 0.0000e+00
  Conclusion   : REJECT H0. Late deliveries significantly degrade customer satisfaction.


## 2. D4 Alignment: Chi-Square for Bad Reviews
**Question:** Is the likelihood of receiving a 1-star "bad review" statistically dependent on whether the delivery was late?
**Dashboard:** D4 (Customer Experience Impact)


In [27]:
contingency_table = pd.crosstab(df['is_bad_review'], df['is_late'])

chi2_stat, chi2_pval, dof, expected = stats.chi2_contingency(contingency_table)

print(f"Chi-Square Test (1-Star Review vs Delivery Lateness)")
print(f"  Chi-Square Statistic : {chi2_stat:.4f}")
print(f"  P-Value              : {chi2_pval:.4e}")
if chi2_pval < 0.05:
    print("  Conclusion           : REJECT H0. Receiving a 1-star review is statistically dependent on late delivery.")
else:
    print("  Conclusion           : Fail to reject H0.")


Chi-Square Test (1-Star Review vs Delivery Lateness)
  Chi-Square Statistic : 10990.6832
  P-Value              : 0.0000e+00
  Conclusion           : REJECT H0. Receiving a 1-star review is statistically dependent on late delivery.


## 3. D2 Alignment: ANOVA for Geographic Variance
**Question:** Do different seller states have statistically significant differences in delivery delay?
**Dashboard:** D2 (Geographic Risk Monitor)


In [28]:
top_5_states = df['seller_state'].value_counts().head(5).index
anova_df = df[df['seller_state'].isin(top_5_states)][['seller_state', 'delivery_delay']].dropna()
state_groups = [group['delivery_delay'].values for _, group in anova_df.groupby('seller_state')]

f_stat, anova_pval = stats.f_oneway(*state_groups)

print(f"ANOVA Test (Delivery Delay Across Top 5 Seller States)")
print(f"  F-Statistic : {f_stat:.4f}")
print(f"  P-Value     : {anova_pval:.4e}")
if anova_pval < 0.05:
    print("  Conclusion  : REJECT H0. Seller state fundamentally influences delivery speed.")
else:
    print("  Conclusion  : Fail to reject H0.")


ANOVA Test (Delivery Delay Across Top 5 Seller States)
  F-Statistic : 397.0750
  P-Value     : 0.0000e+00
  Conclusion  : REJECT H0. Seller state fundamentally influences delivery speed.


## 4. D3 Alignment: Correlation between Seller Delay Rate & Ratings
**Question:** At the individual seller level (minimum 10 orders), does a higher delay rate correlate with lower average review scores?
**Dashboard:** D3 (Seller Performance Scorecard)


In [29]:
MIN_ORDERS = 10
seller_stats = df.groupby('seller_id').agg(
    total_orders=('order_id', 'count'),
    late_orders=('is_late', 'sum'),
    avg_review_score=('review_score', 'mean')
).reset_index()

seller_stats = seller_stats[seller_stats['total_orders'] >= MIN_ORDERS].copy()
seller_stats['delay_rate_pct'] = (seller_stats['late_orders'] / seller_stats['total_orders'] * 100)

corr, p_val = stats.pearsonr(seller_stats['delay_rate_pct'], seller_stats['avg_review_score'])

print(f"Pearson Correlation (Seller Delay Rate vs Seller Avg Review Score, min {MIN_ORDERS} orders)")
print(f"  Correlation coefficient (r) : {corr:.4f}")
print(f"  P-Value                     : {p_val:.4e}")
if p_val < 0.05:
    print("  Conclusion                  : REJECT H0. At the seller level, frequent delays actively harm seller ratings.")
else:
    print("  Conclusion                  : Fail to reject H0.")


Pearson Correlation (Seller Delay Rate vs Seller Avg Review Score, min 10 orders)
  Correlation coefficient (r) : -0.3898
  P-Value                     : 8.6807e-50
  Conclusion                  : REJECT H0. At the seller level, frequent delays actively harm seller ratings.


## 5. D1 Alignment: Overall ETA Accuracy KPI
**Question:** On average, how accurate are the estimated delivery dates?
**Dashboard:** D1 (Executive Pulse)


In [30]:
# Create eta_diff if it wasn't preserved
if 'eta_diff' not in df.columns:
    df['eta_diff'] = df['actual_delivery_days'] - df['estimated_delivery_days']

mean_eta_diff = df['eta_diff'].mean()
print(f"Overall Average ETA Difference (Days): {mean_eta_diff:.2f}")
print("   * Note: A negative number means the package arrived EARLY (ETA had buffer).")
print("   * A positive number means the package arrived LATE (ETA was breached).")



Overall Average ETA Difference (Days): -11.70
   * Note: A negative number means the package arrived EARLY (ETA had buffer).
   * A positive number means the package arrived LATE (ETA was breached).
